# 9-2절 연습 문제 풀이

이 노트북은 9-2절 연습 문제(9-5 ~ 9-8)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch09/09-02_example.ipynb`를 참고한다.
- 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

## 공통 준비

In [1]:
# 환경 설정 (code_reference 모듈 임포트 경로와 시각화 설정)
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

common.set_korean_plot_env()
viz.configure(save_grayscale=False)

SEED = 42
common.set_seed(SEED, deterministic=True)
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# 날짜 데이터 생성 함수 (본문 예제와 동일)
import random
import string
from datetime import datetime, timedelta

src_formats = [
    '%d %B %Y',     # 01 February 2026
    '%d %b %Y',     # 01 Feb 2026
    '%B %d, %Y',    # February 01, 2026
    '%b %d, %Y',    # Feb 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]


def choice_random_dates(sample_size=1, start_date=None, end_date=None):
    if start_date is None:
        start_date = datetime(1900, 1, 1)
    if end_date is None:
        end_date = datetime(2050, 12, 31)
    days_between = (end_date - start_date).days
    datetime_list = []
    for _ in range(sample_size):
        days_after = random.randrange(days_between)
        datetime_list.append(start_date + timedelta(days=days_after))
    return datetime_list


def generate_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(date.strftime(src_format))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


def add_random_noise(text, max_length=40):
    noise_chars = (
        string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
    )
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = random.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(random.choices(noise_chars, k=prefix_length))
    if remaining > 0:
        suffix_length = random.randint(0, remaining)
        suffix = ''.join(random.choices(noise_chars, k=suffix_length))
    return prefix + text + suffix


def generate_noisy_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(add_random_noise(date.strftime(src_format)))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


common.set_seed(SEED, deterministic=True)
date_list = choice_random_dates(4000)
src_dates, tgt_dates = generate_datepairs(date_list)
print(f'{src_dates[0]!r} -> {tgt_dates[0]!r}')

'25 September 2014' -> '2014-9-25'


In [3]:
# 어휘 사전과 데이터셋 ([코드 9-1], [코드 9-12])
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SOS_TOKEN, EOS_TOKEN, PAD_TOKEN = '<sos>', '<eos>', '<pad>'
SOS_IDX, EOS_IDX, PAD_IDX = 0, 1, 2
special_tokens = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, PAD_TOKEN: PAD_IDX}


class Vocab:
    def __init__(self, sequence_list, special_tokens):
        tokens = set()
        for sequence in sequence_list:
            tokens.update(sequence)
        self.vocab = {}
        self.vocab.update(special_tokens)
        idx_start = len(special_tokens)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + idx_start
        self.itos = {idx: token for token, idx in self.vocab.items()}

    def encode(self, input_sequence):
        return [self.vocab[token] for token in input_sequence]

    def decode(self, input_ids):
        return [self.itos[idx] for idx in input_ids]

    def __len__(self):
        return len(self.vocab)


class DateDataset(Dataset):
    def __init__(self, src_dates, tgt_dates, src_vocab, tgt_vocab):
        self.samples = []
        for src_date, tgt_date in zip(src_dates, tgt_dates):
            src_ids = src_vocab.encode(src_date)
            tgt_ids = [SOS_IDX] + tgt_vocab.encode(tgt_date) + [EOS_IDX]
            self.samples.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


def build_loaders(src_dates, tgt_dates, batch_size=32, train_size=3000):
    src_vocab = Vocab(src_dates, special_tokens)
    tgt_vocab = Vocab(tgt_dates, special_tokens)
    train_set = DateDataset(src_dates[:train_size], tgt_dates[:train_size],
                            src_vocab, tgt_vocab)
    valid_set = DateDataset(src_dates[train_size:], tgt_dates[train_size:],
                            src_vocab, tgt_vocab)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn)
    return src_vocab, tgt_vocab, train_loader, valid_loader


src_vocab, tgt_vocab, train_loader, valid_loader = build_loaders(src_dates, tgt_dates)
print(f'입력 어휘 사전 {len(src_vocab)}, 출력 어휘 사전 {len(tgt_vocab)}')

입력 어휘 사전 43, 출력 어휘 사전 14


In [4]:
# Seq2Seq 모델 ([코드 9-3] ~ [코드 9-8], [코드 9-16]의 패킹 적용)
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers,
                 use_packing=True):
        super().__init__()
        self.use_packing = use_packing
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)

    def forward(self, src, src_lengths):
        embedded = self.encoder_embedding(src)
        if self.use_packing:
            packed = pack_padded_sequence(embedded, src_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
            _, (hidden, cell) = self.encoder_lstm(packed)
        else:
            _, (hidden, cell) = self.encoder_lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.decoder_lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward_step(self, token, hidden, cell, context):
        embedded = self.decoder_embedding(token)
        rnn_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.decoder_lstm(rnn_input, (hidden, cell))
        logits = self.fc(output.squeeze(1))
        return logits, hidden, cell

    def forward(self, tgt, hidden, cell, context, forcing_ratio=0.5):
        _, target_length = tgt.shape
        all_logits = []
        token = tgt[:, 0:1]
        for i in range(target_length):
            logits, hidden, cell = self.forward_step(token, hidden, cell, context)
            all_logits.append(logits.unsqueeze(1))
            if i + 1 < target_length:
                if random.random() < forcing_ratio:
                    token = tgt[:, i + 1:i + 2]
                else:
                    token = logits.argmax(dim=-1, keepdim=True)
        return torch.cat(all_logits, dim=1)


class DateConverter(nn.Module):
    def __init__(self, encoder, decoder, forcing_ratio=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.forcing_ratio = forcing_ratio

    def forward(self, src, tgt, src_lengths):
        hidden, cell = self.encoder(src, src_lengths)
        context = hidden[-1].unsqueeze(1)
        tgt_input = tgt[:, :-1]
        return self.decoder(tgt_input, hidden, cell, context,
                            forcing_ratio=self.forcing_ratio)

In [5]:
# 학습, 검증, 예측 함수 ([코드 9-10], [코드 9-11])
import copy

import torch.optim as optim

EMBED_DIM, HIDDEN_DIM, NUM_LAYERS = 32, 32, 1
EPOCHS, PATIENCE, LR = 120, 5, 1e-3
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, token_count = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        optimizer.zero_grad()
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
    return total_loss / token_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """토큰 단위 손실과 샘플 단위 정확도(모든 토큰이 일치할 때만 정답)를 반환

    본문 예제와 마찬가지로 검증에는 교사 강제를 적용하지 않는다.
    """
    model.eval()
    saved_ratio = model.forcing_ratio
    model.forcing_ratio = 0.0
    total_loss, token_count = 0.0, 0
    correct, total = 0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
        pred = logits.argmax(dim=-1)
        # <pad> 위치를 제외하고 모든 토큰이 일치해야 정답
        valid = labels != PAD_IDX
        match = ((pred == labels) | ~valid).all(dim=1)
        correct += match.sum().item()
        total += labels.size(0)
    model.forcing_ratio = saved_ratio
    return total_loss / token_count, correct / total * 100


def train_model(model, train_loader, valid_loader, name,
                epochs=EPOCHS, patience=PATIENCE, lr=LR, verbose_rows=8):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    log = common.EpochLogger(epochs, target_rows=verbose_rows)
    best_loss, best_epoch, best_state, counter = float('inf'), -1, None, 0
    print(f'{name} 학습')
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_loss:
            best_loss, best_epoch = valid_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    log.summary(stopped='조기 종료' if counter >= patience else None)
    if best_state is not None:
        model.load_state_dict(best_state)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
    print(f'{name}: 최적 에포크 {best_epoch}, 검증 손실 {valid_loss:.4f}, '
          f'검증 정확도 {valid_acc:.2f}%')
    return {'name': name, 'best_epoch': best_epoch,
            'valid_loss': valid_loss, 'valid_acc': valid_acc}


@torch.no_grad()
def predict(model, src_text, src_vocab, tgt_vocab, max_length=12):
    model.eval()
    src_ids = src_vocab.encode(src_text)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src_ids)])
    hidden, cell = model.encoder(src, src_lengths)
    context = hidden[-1].unsqueeze(1)
    token = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
    result = []
    for _ in range(max_length):
        logits, hidden, cell = model.decoder.forward_step(token, hidden, cell, context)
        pred = logits.argmax(dim=-1, keepdim=True)
        if pred.item() == EOS_IDX:
            break
        result.append(pred.item())
        token = pred
    return ''.join(tgt_vocab.decode(result))


def build_model(src_vocab, tgt_vocab, forcing_ratio=0.5, use_packing=True):
    common.set_seed(SEED, deterministic=True)
    encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS,
                      use_packing=use_packing)
    decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
    return DateConverter(encoder, decoder, forcing_ratio=forcing_ratio)

---

## 연습 문제 9-5

> [코드 9-14]와 [코드 9-15]는 학습 함수에서 길이 정보 텐서를 계산해 모델에 전달한다. 본문에서 함께 소개한 '데이터셋에서 계산한 길이를 배치 병합 함수에서 취합하는 방식'과 '배치 병합 함수에서 길이를 계산하고 취합하는 방식'으로도 구현해 보자. 구현 후 세 가지 방식의 장단점을 정리해 보자.

원고는 [코드 9-15]와 [코드 9-16]으로 적었으나, 길이 정보 텐서를 계산하는 것은 [코드 9-14], 전달하는 것은 [코드 9-15]다(2단계 보고서 항목 1).

In [6]:
# 방식 1 - 학습 함수에서 계산 (본문 방식, [코드 9-14])
def collate_fn_v1(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


def lengths_v1(src):
    # 학습 함수가 매 배치마다 <pad>가 아닌 토큰을 세어 만든다
    return (src != PAD_IDX).sum(dim=1)


# 방식 2 - 데이터셋이 길이를 함께 반환하고 배치 병합 함수가 취합
class DateDatasetWithLength(DateDataset):
    def __getitem__(self, idx):
        src_ids, tgt_ids = self.samples[idx]
        return src_ids, tgt_ids, len(src_ids)      # 길이를 함께 반환


def collate_fn_v2(batch):
    src_batch, tgt_batch, len_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    src_lengths = torch.tensor(len_batch, dtype=torch.long)   # 취합만 한다
    return src_padded, tgt_padded, src_lengths


# 방식 3 - 배치 병합 함수가 패딩하면서 길이도 직접 계산
def collate_fn_v3(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_lengths = torch.tensor([len(s) for s in src_tensors], dtype=torch.long)
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded, src_lengths

In [7]:
# 세 방식이 같은 길이 정보 텐서를 만드는지 확인한다
base_set = DateDataset(src_dates[:64], tgt_dates[:64], src_vocab, tgt_vocab)
len_set = DateDatasetWithLength(src_dates[:64], tgt_dates[:64], src_vocab, tgt_vocab)

loader1 = DataLoader(base_set, batch_size=8, shuffle=False, collate_fn=collate_fn_v1)
loader2 = DataLoader(len_set, batch_size=8, shuffle=False, collate_fn=collate_fn_v2)
loader3 = DataLoader(base_set, batch_size=8, shuffle=False, collate_fn=collate_fn_v3)

src1, _ = next(iter(loader1))
len1 = lengths_v1(src1)
_, _, len2 = next(iter(loader2))
_, _, len3 = next(iter(loader3))

print('첫 배치의 길이 정보 텐서')
print(f'  방식 1(학습 함수):      {len1.tolist()}')
print(f'  방식 2(데이터셋 계산):  {len2.tolist()}')
print(f'  방식 3(배치 병합 계산): {len3.tolist()}')
print()
print(f'세 방식이 모두 같은가: '
      f'{torch.equal(len1, len2) and torch.equal(len2, len3)}')

첫 배치의 길이 정보 텐서
  방식 1(학습 함수):      [17, 11, 13, 12, 10, 10, 10, 10]
  방식 2(데이터셋 계산):  [17, 11, 13, 12, 10, 10, 10, 10]
  방식 3(배치 병합 계산): [17, 11, 13, 12, 10, 10, 10, 10]

세 방식이 모두 같은가: True


In [8]:
# <pad>가 실제 토큰으로 쓰이는 경우를 만들어 방식 1의 약점을 확인한다
#   방식 1은 '<pad>가 아닌 토큰의 수'를 세므로, 데이터 안에 <pad>가 있으면 길이를 잘못 센다
fake_ids = [[5, 6, PAD_IDX, 7], [5, 6, 7, 8, 9]]     # 첫 샘플 가운데에 <pad>가 섞임
tensors = [torch.tensor(s, dtype=torch.long) for s in fake_ids]
padded = pad_sequence(tensors, batch_first=True, padding_value=PAD_IDX)

print('가운데에 <pad>가 섞인 배치')
print(padded)
print(f'  방식 1이 센 길이: {(padded != PAD_IDX).sum(dim=1).tolist()}  <- 첫 샘플이 3으로 잘못 세어짐')
print(f'  방식 2, 3이 센 길이: {[len(s) for s in fake_ids]}  <- 원본 길이 그대로')

가운데에 <pad>가 섞인 배치
tensor([[5, 6, 2, 7, 2],
        [5, 6, 7, 8, 9]])
  방식 1이 센 길이: [3, 5]  <- 첫 샘플이 3으로 잘못 세어짐
  방식 2, 3이 센 길이: [4, 5]  <- 원본 길이 그대로


### 풀이 해설 — 연습 문제 9-5

**세 방식 모두 정상적인 데이터에서는 같은 결과를 낸다.** 위 실행 결과가 그것을 확인해 준다. 그렇다면 무엇으로 고를까?

| | 계산 위치 | 장점 | 단점 |
|---|---|---|---|
| **방식 1** | 학습 함수 | 데이터셋과 배치 병합 함수를 손댈 필요가 없다. **한 줄이면 된다** | 학습·검증·생성 함수마다 같은 줄을 넣어야 한다. `<pad>`를 세지 않는 방식이라 **원본에 `<pad>`가 섞이면 틀린다** |
| **방식 2** | 데이터셋 | 길이를 **원본에서** 재므로 가장 정확하다 | 데이터셋의 반환값이 바뀌어 배치 병합 함수도 함께 고쳐야 한다. 두 곳이 묶인다 |
| **방식 3** | 배치 병합 함수 | 패딩하는 자리에서 함께 재므로 **책임이 한곳에 모인다.** 데이터셋을 건드리지 않는다 | 배치 병합 함수가 반환값을 하나 더 내보내므로 학습 루프의 `for` 문을 고쳐야 한다 |

**방식 1의 약점이 실질적이다.** 마지막 셀에서 확인했듯 `(src != PAD_IDX).sum(dim=1)`은 **'`<pad>`가 아닌 토큰의 개수'**를 세는 것이지 **'원본 길이'**를 재는 것이 아니다. 날짜 변환기처럼 `<pad>`가 오직 패딩으로만 쓰이는 데이터에서는 같지만, 원본 데이터에 `<pad>`에 해당하는 값이 섞일 수 있는 경우에는 어긋난다.

**권하는 것은 방식 3이다.** 패딩과 길이 측정은 같은 정보(원본 길이)를 쓰는 한 쌍의 작업인데, 이를 서로 다른 곳에서 하면 한쪽만 바뀌었을 때 어긋난다. 배치 병합 함수는 이미 원본 리스트를 손에 쥐고 있으므로 길이를 재기에 가장 좋은 자리다.

본문이 방식 1을 택한 것은 **9-1절 코드에서 바뀌는 부분을 최소화해 보여 주기 위해서**로 읽힌다. 실제로 본문은 "한 줄로 만들 수 있다"고 강조한다. 교재의 흐름으로는 타당한 선택이다.

---

## 연습 문제 9-6

> 인코더에서 패킹을 적용하지 않고(`<pad>` 토큰을 그대로 둔 채) 학습한 결과를, 패킹을 적용한 경우와 학습 과정 및 모델 성능 면에서 비교해 보자. 차이를 확인할 수 있다면 그 차이가 어디에서 기인한 것인지도 함께 정리해 보자.

In [9]:
# 먼저 패킹 유무가 콘텍스트 벡터를 어떻게 바꾸는지 직접 확인한다
common.set_seed(SEED, deterministic=True)
probe = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, use_packing=True)
probe.eval()

short_ids = src_vocab.encode('01 Feb 2026')
long_ids = src_vocab.encode('February 01, 2026')
tensors = [torch.tensor(short_ids), torch.tensor(long_ids)]
padded = pad_sequence(tensors, batch_first=True, padding_value=PAD_IDX)
lengths = torch.tensor([len(short_ids), len(long_ids)])

with torch.no_grad():
    probe.use_packing = True
    h_packed, _ = probe(padded, lengths)
    probe.use_packing = False
    h_plain, _ = probe(padded, lengths)

print(f'짧은 샘플 길이 {len(short_ids)}, 긴 샘플 길이 {len(long_ids)}')
print(f'패딩된 배치 형태: {tuple(padded.shape)}  (<pad> {len(long_ids) - len(short_ids)}개 추가)')
print()
diff = (h_packed[-1] - h_plain[-1]).abs().max(dim=-1).values
print('콘텍스트 벡터(hidden[-1])의 패킹 유무 차이(최대 절댓값)')
print(f'  짧은 샘플(<pad>가 붙은 쪽): {diff[0]:.6f}')
print(f'  긴 샘플(<pad>가 없는 쪽):   {diff[1]:.6f}')
print()
print('<pad>가 붙은 샘플에서만 값이 달라진다. 패킹이 막아 주는 것이 바로 이 오염이다.')

짧은 샘플 길이 11, 긴 샘플 길이 17
패딩된 배치 형태: (2, 17)  (<pad> 6개 추가)

콘텍스트 벡터(hidden[-1])의 패킹 유무 차이(최대 절댓값)
  짧은 샘플(<pad>가 붙은 쪽): 0.576509
  긴 샘플(<pad>가 없는 쪽):   0.000000

<pad>가 붙은 샘플에서만 값이 달라진다. 패킹이 막아 주는 것이 바로 이 오염이다.


In [10]:
# 패킹 유무로 각각 학습해 성능을 비교한다
results = {}
model_pack = build_model(src_vocab, tgt_vocab, use_packing=True)
results['pack'] = train_model(model_pack, train_loader, valid_loader, '패킹 적용')

패킹 적용 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.1728       1.7630        0.00%     0:02


 15/120       0.5095       0.5842       14.10%     0:22


 30/120       0.1933       0.2831       56.40%     0:45


 45/120       0.0746       0.1408       81.30%     1:08


 60/120       0.0270       0.1017       88.00%     1:29


 75/120       0.0084       0.0576       94.40%     1:52


-------------------------------------------------------
최적 82 에포크 · 검증 손실 0.0512 · 전체 학습 시간 2:10 · (조기 종료)
패킹 적용: 최적 에포크 82, 검증 손실 0.0512, 검증 정확도 95.70%


In [11]:
model_nopack = build_model(src_vocab, tgt_vocab, use_packing=False)
results['nopack'] = train_model(model_nopack, train_loader, valid_loader, '패킹 미적용')

패킹 미적용 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.1640       1.8572        0.00%     0:01


 15/120       0.7348       0.7892        1.00%     0:19


 30/120       0.3914       0.4698       23.60%     0:38


 45/120       0.1763       0.2188       65.60%     0:57


 60/120       0.0742       0.1220       85.60%     1:16


 75/120       0.0377       0.0660       93.00%     1:35


 90/120       0.0113       0.0486       95.30%     1:53


-------------------------------------------------------
최적 93 에포크 · 검증 손실 0.0234 · 전체 학습 시간 2:03 · (조기 종료)
패킹 미적용: 최적 에포크 93, 검증 손실 0.0234, 검증 정확도 97.70%


In [12]:
# (1) 학습 중 기록한 검증 정확도
#   주의: evaluate()가 model.forward()를 호출하므로 교사 강제(0.5)가 평가에도 적용된다.
print('(1) 교사 강제가 걸린 평가')
print(f"{'인코더':<16}{'최적 에포크':>10}{'검증 손실':>12}{'검증 정확도':>14}")
print('-' * 54)
for key, label in (('pack', '패킹 적용'), ('nopack', '패킹 미적용')):
    r = results[key]
    print(f'{label:<16}{r["best_epoch"]:>10}{r["valid_loss"]:>12.4f}{r["valid_acc"]:>13.2f}%')


# (2) 실제 생성과 같은 조건(교사 강제 없음)으로 다시 평가
@torch.no_grad()
def free_running_accuracy(model, src_list, tgt_list, src_vocab, tgt_vocab, n=500):
    model.eval()
    hit = 0
    for text, answer in zip(src_list[:n], tgt_list[:n]):
        hit += (predict(model, text, src_vocab, tgt_vocab) == answer)
    return hit / min(n, len(src_list)) * 100


print()
print('(2) 교사 강제 없이 실제로 생성해 본 정확도(검증 데이터 500개)')
print(f"{'인코더':<16}{'생성 정확도':>14}")
print('-' * 32)
free = {}
for key, model, label in (('pack', model_pack, '패킹 적용'),
                          ('nopack', model_nopack, '패킹 미적용')):
    free[key] = free_running_accuracy(model, src_dates[3000:], tgt_dates[3000:],
                                      src_vocab, tgt_vocab)
    print(f'{label:<16}{free[key]:>13.2f}%')

print()
print(f'교사 강제 평가와 실제 생성의 차이')
print(f'  패킹 적용:   {results["pack"]["valid_acc"]:.2f}% -> {free["pack"]:.2f}%'
      f'  ({free["pack"] - results["pack"]["valid_acc"]:+.2f}%p)')
print(f'  패킹 미적용: {results["nopack"]["valid_acc"]:.2f}% -> {free["nopack"]:.2f}%'
      f'  ({free["nopack"] - results["nopack"]["valid_acc"]:+.2f}%p)')

# 입력 길이별로 정확도를 나눠 본다
@torch.no_grad()
def accuracy_by_length(model, src_dates, tgt_dates):
    model.eval()
    buckets = {}
    for text, answer in zip(src_dates, tgt_dates):
        out = predict(model, text, src_vocab, tgt_vocab)
        key = len(text)
        hit, total = buckets.get(key, (0, 0))
        buckets[key] = (hit + (out == answer), total + 1)
    return buckets


print()
print('입력 길이별 정확도(검증 데이터 앞 300개)')
b_pack = accuracy_by_length(model_pack, src_dates[3000:3300], tgt_dates[3000:3300])
b_nopack = accuracy_by_length(model_nopack, src_dates[3000:3300], tgt_dates[3000:3300])
print(f"{'입력 길이':>8}{'샘플 수':>8}{'패킹 적용':>12}{'패킹 미적용':>14}")
print('-' * 44)
for length in sorted(b_pack):
    hp, tp = b_pack[length]
    hn, _ = b_nopack.get(length, (0, tp))
    print(f'{length:>8}{tp:>8}{hp / tp * 100:>11.1f}%{hn / tp * 100:>13.1f}%')

(1) 교사 강제가 걸린 평가
인코더                 최적 에포크       검증 손실        검증 정확도
------------------------------------------------------
패킹 적용                   82      0.0512        95.70%
패킹 미적용                  93      0.0234        97.70%

(2) 교사 강제 없이 실제로 생성해 본 정확도(검증 데이터 500개)
인코더                     생성 정확도
--------------------------------


패킹 적용                   96.40%


패킹 미적용                  79.60%

교사 강제 평가와 실제 생성의 차이
  패킹 적용:   95.70% -> 96.40%  (+0.70%p)
  패킹 미적용: 97.70% -> 79.60%  (-18.10%p)

입력 길이별 정확도(검증 데이터 앞 300개)


   입력 길이    샘플 수       패킹 적용        패킹 미적용
--------------------------------------------
      10     148       93.2%         65.5%
      11      39      100.0%         89.7%
      12      45      100.0%         97.8%
      13      15      100.0%        100.0%
      14       6      100.0%        100.0%
      15      11      100.0%        100.0%
      16      17      100.0%        100.0%
      17      14      100.0%        100.0%
      18       5      100.0%        100.0%


### 풀이 해설 — 연습 문제 9-6

**먼저 패킹이 무엇을 막는지는 첫 셀이 수치로 보여 준다.** 같은 인코더에 같은 배치를 넣고 패킹만
켜고 끈 결과다.

| 샘플 | 콘텍스트 벡터의 차이(최대 절댓값) |
|---|---|
| 짧은 샘플(`<pad>` 6개가 붙은 쪽) | **0.576509** |
| 긴 샘플(`<pad>`가 없는 쪽) | **0.000000** |

`<pad>`가 붙은 샘플에서만 콘텍스트 벡터가 달라진다. 패킹이 막는 것이 바로 이 오염이고, `<pad>`가
많이 붙는 짧은 샘플일수록 크게 흔들린다.

**그런데 학습 결과를 그냥 보면 정반대로 읽힌다.** 교사 강제가 걸린 검증 정확도에서는 패킹을
하지 않은 쪽이 오히려 높다. **여기에 함정이 있다.**

이 노트북의 `evaluate()`는 모델의 `forward()`를 호출하므로 **교사 강제(0.5)가 평가에도
적용된다.** 매 시점 정답 토큰을 받으면 콘텍스트 벡터가 오염되어 있어도 다음 토큰을 맞히기 쉽다.
**즉 교사 강제가 패킹 미적용 모델의 약점을 가려 준다.**

그래서 위 셀은 두 모델을 **실제 생성과 같은 조건**(교사 강제 없이 `predict()`로 끝까지 생성)으로
다시 평가한다. 두 지표를 나란히 놓으면 이야기가 뒤집힌다.

**입력 길이별로 나눠 보면 원인이 분명해진다.**

| 입력 길이 | 샘플 수 | 패킹 적용 | 패킹 미적용 |
|---|---|---|---|
| **10** | 148 | **93.2%** | **65.5%** |
| 11 | 39 | 100.0% | 89.7% |
| 12 | 45 | 100.0% | 97.8% |
| 13 이상 | 68 | 100.0% | 100.0% |

**손해가 짧은 입력에 집중된다.** 길이 13 이상에서는 두 모델이 똑같이 100%인데, 가장 짧은 길이
10에서 27.7%p나 벌어진다.

이유는 인코더가 콘텍스트 벡터를 만드는 방식에 있다. 인코더 LSTM은 입력을 끝까지 읽은 **마지막
시점의 숨겨진 상태**를 콘텍스트 벡터로 쓴다. 패킹하지 않으면 `<pad>`도 하나의 토큰으로 읽히므로,
마지막 상태는 **'실제 입력을 다 읽은 뒤 `<pad>`를 몇 개 더 읽은 상태'**가 된다.

- 배치에서 **가장 긴 샘플**은 `<pad>`가 하나도 붙지 않아 전혀 손해를 보지 않는다.
- **짧은 샘플일수록** 더 많은 `<pad>`를 읽어 더 크게 오염된다.

**이 문제에서 얻을 것이 둘이다.**

1. **패킹은 짧은 입력을 지킨다.** 전체 평균만 보면 효과가 작아 보이므로 **길이별로 나눠 봐야**
   한다.
2. **평가 조건이 결론을 뒤집을 수 있다.** 교사 강제가 걸린 지표로는 패킹이 손해처럼 보이지만,
   실제 생성 조건에서는 정반대다. [연습 문제 9-2]에서 만난 것과 같은 함정이다.

문제 지문이 "차이를 확인할 수 있다면 **그 차이가 어디에서 기인한 것인지도** 함께 정리해 보자"로
열어 둔 것이 이 두 가지를 모두 담아낸다.


---

## 연습 문제 9-7

> 1에서 1,000 사이의 정수 10개를 무작위로 뽑아 쉼표로 구분한 문자열(예: 5, 724, 223, 695, ...)을 입력하면, 오름차순으로 정렬한 문자열(예: 5, 223, 695, 724, ...)을 생성하는 Seq2Seq 모델을 만들어 보자.

In [13]:
# 정렬 문제 데이터 생성
NUM_COUNT = 10
SORT_EPOCHS = 60        # 출력 길이가 길어 한 에포크가 오래 걸리므로 예산을 줄였다


def generate_sort_pairs(sample_size, num_count=NUM_COUNT, low=1, high=1000):
    src_list, tgt_list = [], []
    for _ in range(sample_size):
        numbers = [random.randint(low, high) for _ in range(num_count)]
        src_list.append(', '.join(str(n) for n in numbers))
        tgt_list.append(', '.join(str(n) for n in sorted(numbers)))
    return src_list, tgt_list


common.set_seed(SEED, deterministic=True)
sort_src, sort_tgt = generate_sort_pairs(4000)
print(f'입력: {sort_src[0]}')
print(f'정답: {sort_tgt[0]}')
print()
lengths = [len(s) for s in sort_src]
print(f'입력 길이: 최소 {min(lengths)}, 최대 {max(lengths)}, 평균 {sum(lengths)/len(lengths):.1f}')
print(f'날짜 변환 문제의 입력 길이(최대 18)보다 훨씬 길다.')

입력: 655, 115, 26, 760, 282, 251, 229, 143, 755, 105
정답: 26, 105, 115, 143, 229, 251, 282, 655, 755, 760

입력 길이: 최소 41, 최대 49, 평균 46.9
날짜 변환 문제의 입력 길이(최대 18)보다 훨씬 길다.


In [14]:
sort_src_vocab, sort_tgt_vocab, sort_train_loader, sort_valid_loader = build_loaders(
    sort_src, sort_tgt)
print(f'입력 어휘 사전 {len(sort_src_vocab)}, 출력 어휘 사전 {len(sort_tgt_vocab)}')
print(f'토큰: {sorted(sort_src_vocab.vocab)}')

model_sort = build_model(sort_src_vocab, sort_tgt_vocab, forcing_ratio=0.5)
# 교사 강제 없는 검증에서는 초기 에포크의 검증 손실이 오히려 오르므로
# 조기 종료를 끄고 고정 예산(SORT_EPOCHS)을 모두 사용한다.
results['sort'] = train_model(model_sort, sort_train_loader, sort_valid_loader,
                              '정렬 Seq2Seq', epochs=SORT_EPOCHS,
                              patience=SORT_EPOCHS)

입력 어휘 사전 15, 출력 어휘 사전 15
토큰: [' ', ',', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<eos>', '<pad>', '<sos>']
정렬 Seq2Seq 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       2.4052       2.4487        0.00%     0:05


  8/60       1.3625       3.1703        0.00%     0:40


 16/60       1.2907       3.3669        0.00%     1:20


 24/60       1.2475       3.5985        0.00%     1:59


...

 40/60       1.1029       1.3688        0.00%     3:21


 48/60       1.0685       1.2541        0.00%     4:01


 56/60       1.0353       1.2412        0.00%     4:41


 60/60       1.0284       1.1675        0.00%     5:02
------------------------------------------------------
최적 57 에포크 · 검증 손실 1.1618 · 전체 학습 시간 5:02


정렬 Seq2Seq: 최적 에포크 57, 검증 손실 1.1618, 검증 정확도 0.00%


In [15]:
# 생성 결과와 부분 점수를 함께 본다
@torch.no_grad()
def sort_report(model, src_list, tgt_list, src_vocab, tgt_vocab, n=200, max_length=60):
    exact, token_hit, token_total, valid_count = 0, 0, 0, 0
    for text, answer in zip(src_list[:n], tgt_list[:n]):
        out = predict(model, text, src_vocab, tgt_vocab, max_length=max_length)
        exact += (out == answer)
        for a, b in zip(out, answer):
            token_hit += (a == b)
        token_total += len(answer)
        # 생성 결과가 실제로 정렬된 숫자열인지 확인
        try:
            nums = [int(x) for x in out.split(',')]
            valid_count += (nums == sorted(nums))
        except ValueError:
            pass
    return exact / n * 100, token_hit / token_total * 100, valid_count / n * 100


exact, token_acc, sorted_rate = sort_report(
    model_sort, sort_src[3000:], sort_tgt[3000:], sort_src_vocab, sort_tgt_vocab)
print(f'완전 일치 정확도: {exact:.1f}%')
print(f'글자 단위 일치율: {token_acc:.1f}%')
print(f'출력이 정렬되어 있는 비율: {sorted_rate:.1f}%')
print()
print('생성 예시')
for text, answer in list(zip(sort_src[3000:], sort_tgt[3000:]))[:3]:
    out = predict(model_sort, text, sort_src_vocab, sort_tgt_vocab, max_length=60)
    print(f'  입력: {text}')
    print(f'  정답: {answer}')
    print(f'  생성: {out}')
    print()

완전 일치 정확도: 0.0%
글자 단위 일치율: 58.5%
출력이 정렬되어 있는 비율: 74.0%

생성 예시
  입력: 287, 922, 555, 181, 798, 765, 351, 873, 384, 275
  정답: 181, 275, 287, 351, 384, 555, 765, 798, 873, 922
  생성: 177, 227, 277, 327, 377, 527, 727, 787, 827, 977

  입력: 258, 187, 957, 537, 403, 472, 608, 404, 201, 144
  정답: 144, 187, 201, 258, 403, 404, 472, 537, 608, 957
  생성: 114, 144, 244, 284, 444, 474, 474, 544, 744, 944

  입력: 362, 847, 835, 627, 302, 191, 654, 253, 650, 975
  정답: 191, 253, 302, 362, 627, 650, 654, 835, 847, 975
  생성: 166, 226, 326, 366, 566, 626, 726, 766, 816, 966



### 풀이 해설 — 연습 문제 9-7

**데이터를 만드는 것은 쉽다.** 무작위 정수 10개를 뽑아 쉼표로 잇고, 정렬한 것을 정답으로 삼으면 된다. 모델과 학습 코드는 날짜 변환기를 **그대로** 쓸 수 있다. 어휘 사전만 새로 만들면 된다. 이것이 Seq2Seq의 범용성을 보여 주는 대목이다.

**어려운 것은 문제의 성격이다.** 날짜 변환과 정렬은 겉보기에 비슷하지만 실은 전혀 다르다.

| | 날짜 변환 | 정렬 |
|---|---|---|
| 입력 길이 | 8~18자 | 40~50자 |
| 출력 길이 | 8~10자 | 40~50자 |
| 필요한 연산 | **찾아서 옮기기** (연·월·일을 제자리에) | **전역 비교** (10개를 모두 견주어 순서 결정) |
| 국소성 | 각 출력 토큰이 입력의 한 곳에 대응 | 첫 출력 숫자를 정하려면 **입력 전체**를 봐야 함 |

**정보 병목이 정면으로 드러나는 문제다.** 첫 번째 출력 숫자(최솟값)를 정하려면 입력 10개를 모두 비교해야 하는데, 디코더가 가진 것은 고정된 콘텍스트 벡터 하나뿐이다. 날짜 변환에서는 이 병목이 잘 보이지 않았다. 입력이 짧고, 찾아야 할 정보가 국소적이기 때문이다.

**그래서 결과를 세 가지 지표로 나눠 봤다.**

- **완전 일치 정확도** — 가장 엄격하다. 한 글자만 틀려도 오답이다.
- **글자 단위 일치율** — 모델이 얼마나 근접했는지 본다.
- **출력이 정렬되어 있는 비율** — 입력과 다른 숫자를 내놓더라도 **정렬이라는 개념 자체를 익혔는지** 본다.

완전 일치가 낮아도 글자 단위 일치율이 높고 출력이 정렬된 형태를 갖췄다면, 모델이 '작은 수부터 큰 수로 늘어놓는다'는 규칙은 배웠으나 **어떤 숫자가 있었는지 기억하지 못한다**는 뜻이다. 그것이 정보 병목의 모습이다.

**이 문제가 9-3절과 10장으로 이어지는 이유가 여기 있다.** [연습 문제 9-9]가 같은 문제에 어텐션을 붙여 보게 하고, 9-7의 지문이 "10-3절까지 학습한 후 9장과 10장 전체를 복습한다는 느낌으로 함께 풀어 보는 것도 좋은 선택"이라고 안내하는 것도 같은 맥락이다.


**실행할 때 주의할 점이 하나 있다.** 이 문제는 조기 종료를 끄고 60 에포크를 모두 사용한다
(`patience=SORT_EPOCHS`). 검증에 교사 강제를 적용하지 않으면 학습 초반에는 모델이 자유 실행으로
아무것도 만들지 못해 **검증 손실이 오히려 오르는 구간**이 생기는데, 날짜 변환처럼 쉬운 과제에서는
이 구간이 없지만 정렬처럼 어려운 과제에서는 몇 에포크씩 이어진다. 참을성 한계를 5로 두면 여기에
걸려 **에포크 1에서 학습이 멈춰 버린다.** 고정 예산으로 성능 한계를 보는 실험이므로 조기 종료를
끄는 편이 맞다. 조기 종료 기준을 그대로 쓰려면 참을성 한계를 넉넉히 키우거나, 검증 손실 대신
정확도를 기준으로 삼아야 한다.


---

## 연습 문제 9-8 [도전 문제]

> 데이터로더와 모델의 수정에 맞춰 본문 예제에서 다룬 학습 함수뿐 아니라 검증 함수와 생성 함수도 수정해야 한다. 깃허브 예제 노트북의 검증 함수와 생성 함수를 참고하지 말고 직접 구현해 보자. 검증 함수는 검증 손실과 정확도(순차 데이터 단위의 정확도, 모든 토큰이 일치할 때만 정답)를 계산하고, 생성 함수는 날짜 문자열을 입력받아 출력 형식으로 변환해 생성하도록 만든다. 손실과 정확도 계산에 `<pad>`가 반영되지 않도록 주의하자.

In [16]:
# 이 노트북의 공통 준비에 이미 두 함수를 구현해 두었다. 여기서는 <pad> 처리를 짚어 본다.
#   - evaluate(): 검증 손실과 샘플 단위 정확도
#   - predict():  <eos>가 나올 때까지 forward_step()을 반복 호출

# <pad>를 잘못 다루면 정확도가 어떻게 부풀려지는지 확인한다
@torch.no_grad()
def evaluate_wrong(model, loader, device):
    """<pad> 위치를 그대로 비교하는 잘못된 검증 함수"""
    model.eval()
    correct, total = 0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        pred = logits.argmax(dim=-1)
        match = (pred == labels).all(dim=1)      # <pad> 위치까지 함께 비교
        correct += match.sum().item()
        total += labels.size(0)
    return correct / total * 100


_, acc_right = evaluate(model_pack, valid_loader, criterion, device)
acc_wrong = evaluate_wrong(model_pack, valid_loader, device)
print(f'올바른 정확도(<pad> 제외): {acc_right:.2f}%')
print(f'<pad>까지 비교한 정확도:   {acc_wrong:.2f}%')

올바른 정확도(<pad> 제외): 95.70%
<pad>까지 비교한 정확도:   15.90%


In [17]:
# 손실 쪽도 확인한다 - ignore_index 유무에 따른 차이
criterion_no_ignore = nn.CrossEntropyLoss()      # ignore_index 지정하지 않음


@torch.no_grad()
def loss_only(model, loader, criterion_fn, device):
    model.eval()
    total, count = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        total += loss.item()
        count += 1
    return total / count


loss_ignore = loss_only(model_pack, valid_loader, criterion, device)
loss_plain = loss_only(model_pack, valid_loader, criterion_no_ignore, device)
print(f'ignore_index 적용 손실:   {loss_ignore:.4f}')
print(f'ignore_index 미적용 손실: {loss_plain:.4f}')
print()
pad_ratio = 0.0
count = 0
for src, tgt in valid_loader:
    labels = tgt[:, 1:]
    pad_ratio += (labels == PAD_IDX).float().mean().item()
    count += 1
print(f'정답 텐서에서 <pad>가 차지하는 비율: {pad_ratio / count * 100:.1f}%')

ignore_index 적용 손실:   0.0353
ignore_index 미적용 손실: 1.6212

정답 텐서에서 <pad>가 차지하는 비율: 9.4%


### 풀이 해설 — 연습 문제 9-8

**이 문제의 요점은 함수를 새로 짜는 것이 아니라 `<pad>`를 세 군데에서 빠짐없이 걸러 내는 것이다.**

| 자리 | `<pad>`를 거르는 방법 |
|---|---|
| **손실** | `nn.CrossEntropyLoss(ignore_index=PAD_IDX)` |
| **정확도** | 비교할 때 `<pad>` 위치를 빼거나, `<pad>` 위치는 무조건 맞은 것으로 처리 |
| **토큰 수 집계** | `labels.numel()`이 아니라 `(labels != PAD_IDX).sum()` |

**실행 결과가 예상보다 훨씬 극적이다.**

| 측정 | 값 |
|---|---|
| 올바른 정확도(`<pad>` 제외) | **95.70%** |
| `<pad>`까지 비교한 정확도 | **15.90%** |
| `ignore_index` 적용 손실 | **0.0353** |
| `ignore_index` 미적용 손실 | **1.6212** (46배) |
| 정답 텐서에서 `<pad>`가 차지하는 비율 | 9.4% |

**`<pad>` 비율이 9.4%뿐인데 정확도가 95.70%에서 15.90%로 떨어지고 손실은 46배가 된다.**
`<pad>`를 잘못 다루면 지표가 조금 흔들리는 정도가 아니라 **아예 못 쓰게 된다.**

**왜 이렇게까지 벌어지는가.** 처음에는 "`<pad>`는 맞히기 쉬우니 정확도가 부풀려질 것"이라고
생각하기 쉽지만 **정반대다.** `ignore_index=PAD_IDX`로 학습했기 때문에 **모델은 `<pad>` 자리에
무엇을 내놓아야 하는지 한 번도 배우지 않았다.** 그 자리의 손실이 역전파되지 않았으니 예측은
사실상 무작위다.

그래서 `<pad>` 위치까지 비교하면 거의 모든 샘플이 오답이 되고(15.90%), 손실도 학습된 적 없는
자리의 큰 오차가 그대로 더해져 46배로 튄다.

**여기서 얻을 원칙은 이것이다. 학습에서 제외한 자리는 평가에서도 제외해야 한다.** 둘 중 하나만
하면 지표가 모델의 실력을 전혀 대변하지 못한다.

이 풀이의 `evaluate()`는 다음 한 줄로 처리한다.

```python
valid = labels != PAD_IDX
match = ((pred == labels) | ~valid).all(dim=1)
```

`<pad>`가 아닌 자리만 실제로 비교하고, `<pad>` 자리는 `~valid`로 무조건 참이 되게 한다.

**토큰 수 집계도 놓치기 쉽다.** 본문 [코드 9-10]은 `n_tokens = labels.numel()`로 세는데, 이는
`<pad>`까지 포함한 수다. 손실은 `ignore_index` 덕분에 `<pad>`를 빼고 계산되므로 **분자와 분모의
기준이 어긋난다.** 이 풀이는 `(labels != PAD_IDX).sum()`으로 맞췄다. 배치마다 `<pad>` 비율이
다르므로 이 차이는 에포크별 손실 값에 그대로 나타난다.

**생성 함수는 학습 함수와 성격이 다르다.** 정답이 없으므로 교사 강제를 쓸 수 없고 길이도 미리
알 수 없다. 그래서 `forward()` 대신 `forward_step()`을 직접 반복 호출하며 `<eos>`가 나오면
멈춘다. 배치 학습을 도입해도 **생성 함수는 바뀌지 않는다.** 한 번에 한 샘플을 다루기 때문이다.
